# Train WhisperWatch on Google Colab

Whisper Large V3 writes transcripts. WhisperWatch is the smaller model we train to score those transcripts for human review. The dataset already contains Large V3 transcripts and measurements, so training WhisperWatch needs only a **CPU**.

Open this notebook at [Google Colab](https://colab.research.google.com/). Choose **Runtime → Change runtime type → CPU** and connect to Google's hosted runtime. Run each cell using the play button beside it. Steps 1–5 train, verify, and download the results. Step 6 tries an audio upload on a GPU.

**Already trained logistic regression?** Run steps 1 and 2 with the updated code ZIP, then go directly to **step 7** for boosted trees.

## 1. Install the classifier dependencies

The scikit-learn and joblib versions match the preliminary model. The pandas and PyArrow versions satisfy Colab's package requirements. If you ran the old install cell or Colab asks you to restart after installing, choose **Runtime → Restart session** and continue with step 2.

In [ ]:
%pip install -q scikit-learn==1.8.0 joblib==1.5.3 pandas==2.2.3 "pyarrow>=23.0.1"

## 2. Upload the project code

The repository is private. Use a ZIP of the local checkout's Python source and keep the notebook private. GitHub sign-in stays with GitHub and the local checkout. The Colab session needs zero GitHub credentials.

If you already have `whisperwatch-code.zip`, select it when the upload button appears. To recreate it, open PowerShell in the repository on `feature/model-asr` and run this command.

```powershell
tar.exe -a -cf whisperwatch-code.zip src/__init__.py src/features.py src/train.py src/inference.py src/asr.py
```

This ZIP contains the Python source for the training and audio workflow. Keep it outside Git or remove the ZIP from your checkout after uploading.

In [ ]:
import os
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

PROJECT = Path("/content/Whisperwatch")
SOURCE_FILES = (
    "src/__init__.py", "src/features.py", "src/train.py",
    "src/inference.py", "src/asr.py",
)

print("Choose the WhisperWatch code ZIP (whisperwatch-boosted-code.zip for step 7).")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one ZIP containing the WhisperWatch source files.")
with ZipFile(BytesIO(next(iter(uploaded.values())))) as project_zip:
    missing = set(SOURCE_FILES) - set(project_zip.namelist())
    if missing:
        raise ValueError(f"Missing from ZIP: {', '.join(sorted(missing))}. Use the command above.")
    for name in SOURCE_FILES:
        destination = PROJECT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(project_zip.read(name))

os.chdir(PROJECT)
print("WhisperWatch source is ready.")

## 3. Train on English clips and reload the model

Choose `"full"` for all 28 English training files, roughly 1.1 million rows before filtering and about 460 MB of compressed training metadata. Choose `"baseline"` to reproduce the original 15,000-clip setup. Choose `"load"` to upload a trusted `whisperwatch.joblib` from an earlier run and skip training.

The [pinned Whispered release](https://huggingface.co/datasets/burakaydinofficial/Whispered/tree/25299fd560dde4271f5177118354af5a961b92fd) supplies the public files. We use English clips up to 30 seconds from archive pipeline 0.2.0 with valid word error rates. Training extracts ten numbers from the Whisper transcript, segments, and duration. A clip is labeled high error when WER exceeds 20%. Development data sets the review cutoff. The official test split is reserved for the next step and the overlapping `other` split is excluded.

Whispered and its Common Voice 17 source are documented as CC0-1.0. Cite Whispered (burakaydinofficial, 2026) and [Common Voice (Ardila et al., 2020)](https://aclanthology.org/2020.lrec-1.520/).

In [ ]:
from src.inference import MODEL_PATH, load_model
from src.train import OUTPUT_DIR, train_development

MODE = "full"  # "full", "baseline", or "load"

if MODE == "load":
    print("Choose the whisperwatch.joblib file from a trusted previous run.")
    uploaded = files.upload()
    if set(uploaded) != {"whisperwatch.joblib"}:
        raise ValueError("Upload one file named whisperwatch.joblib.")
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    MODEL_PATH.write_bytes(uploaded["whisperwatch.joblib"])
elif MODE in {"baseline", "full"}:
    train_development(full_training=MODE == "full")
else:
    raise ValueError('Set MODE to "full", "baseline", or "load".')

artifact = load_model()
if "training_rows" in artifact:
    print(f"Training clips used: {artifact['training_rows']:,}")
print(f"Review threshold: {artifact['review_threshold']:.6f}")

## 4. Score a real clip and optionally export the test set

The first cell scores an archived development example. The score helps rank clips for review. The second cell is for final evaluation. After you have decided on the model and procedure using development results, set `MODEL_FROZEN = True` and run it. The baseline's test results already exist, so comparisons on this same test set count as a re-evaluation.

In [ ]:
import pyarrow.parquet as pq

from src.features import extract_features
from src.inference import predict_from_features
from src.train import DATA_DIR, get_shard

dev_file = get_shard("dev", 0, DATA_DIR)
columns = ["whisper_text", "segments", "duration_s"]
batch = next(pq.ParquetFile(dev_file).iter_batches(batch_size=1, columns=columns))
clip = batch.to_pylist()[0]
features = extract_features(clip["segments"], clip["whisper_text"], clip["duration_s"])
result = predict_from_features(features)
print(clip["whisper_text"])
print(f"Review score = {result['review_score']:.3f} ({result['review_label']})")

In [ ]:
from src.train import predict_test

MODEL_FROZEN = False
if MODEL_FROZEN:
    predict_test()
else:
    print("Test scoring is off. Set MODEL_FROZEN = True when ready.")

## 5. Download the model and predictions

Run this cell before disconnecting or switching runtimes. It downloads a ZIP with the model and any CSVs generated in this run. Files in the hosted runtime are temporary. Keep the ZIP with your experiment notes.

In [ ]:
from zipfile import ZIP_DEFLATED

results_zip = PROJECT.parent / "whisperwatch-results.zip"
with ZipFile(results_zip, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(MODEL_PATH, "models/whisperwatch.joblib")
    if MODE in {"baseline", "full"}:
        archive.write(OUTPUT_DIR / "dev_predictions.csv", "outputs/dev_predictions.csv")
    if MODEL_FROZEN:
        archive.write(OUTPUT_DIR / "test_predictions.csv", "outputs/test_predictions.csv")
files.download(str(results_zip))

## 6. Optional live audio with Whisper Large V3

Download your model first. Switch to a hosted **GPU** runtime via **Runtime → Change runtime type**, then run steps 1 and 2 again. In step 3, use `MODE = "load"` and upload the `whisperwatch.joblib` file you extracted from your results ZIP.

Run both cells below. Choose a short English recording (up to 30 seconds). Large V3 downloads about 3 GB of pretrained weights on its first use. It runs in float16 here. The archive does not publish complete decoding settings, so live scores are experimental. [faster-whisper CUDA requirements](https://github.com/SYSTRAN/faster-whisper#gpu)

In [ ]:
%pip install -q faster-whisper==1.2.1 av==16.0.1 ctranslate2==4.9.0 nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

In [ ]:
import subprocess
import sys
from importlib.util import find_spec

subprocess.run(["nvidia-smi", "-L"], check=True)
print("Choose an English audio clip (up to 30 seconds).")
uploaded_audio = files.upload()
if len(uploaded_audio) != 1:
    raise ValueError("Choose one audio file.")
name, audio_bytes = next(iter(uploaded_audio.items()))
audio_path = PROJECT / "uploads" / Path(name).name
audio_path.parent.mkdir(exist_ok=True)
audio_path.write_bytes(audio_bytes)

gpu_env = os.environ.copy()
gpu_env["WHISPERWATCH_ASR_DEVICE"] = "cuda"
gpu_env["WHISPERWATCH_ASR_COMPUTE_TYPE"] = "float16"
library_dirs = [
    str(Path(next(iter(find_spec(package).submodule_search_locations))) / "lib")
    for package in ("nvidia.cublas", "nvidia.cudnn")
]
gpu_env["LD_LIBRARY_PATH"] = ":".join(filter(None, [
    *library_dirs, gpu_env.get("LD_LIBRARY_PATH"),
]))
subprocess.run(
    [sys.executable, "-c",
     "import json, sys; from src.asr import analyze_audio; "
     "print(json.dumps(analyze_audio(sys.argv[1]), indent=2))",
     str(audio_path)],
    cwd=PROJECT, env=gpu_env, check=True,
)

## 7. Compare boosted trees with your saved logistic model

Use a hosted **CPU** runtime. After steps 1 and 2 with the updated notebook and code ZIP, run the first cell below. Upload the original `whisperwatch-results.zip` you downloaded from the **full** logistic regression run. The comparison reads the development CSV inside that ZIP.

The new model trains on the same full English training set with the same ten inputs. It uses small trees and can stop early based on a portion of the training data. Development selects the review cutoff and compares boosted trees, logistic regression, and Whisper confidence. Each method gets the same top 20% review budget. Look at `high_error_caught` to see which method finds the most bad clips. Official test scoring stays off during this comparison.

In [ ]:
from src.train import (
    BOOSTED_MODEL_PATH, BOOSTED_OUTPUT_DIR, train_boosted_development,
)

print("Choose whisperwatch-results.zip from the full logistic regression run.")
uploaded_baseline = files.upload()
if len(uploaded_baseline) != 1:
    raise ValueError("Upload one ZIP from the logistic regression run.")
baseline_path = PROJECT / "logistic_baseline.zip"
baseline_path.write_bytes(next(iter(uploaded_baseline.values())))
train_boosted_development(baseline_path)

Run the next cell to download the boosted model, its development predictions, and the comparison table.

In [ ]:
from zipfile import ZIP_DEFLATED

boosted_zip = PROJECT.parent / "whisperwatch-boosted-results.zip"
with ZipFile(boosted_zip, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(BOOSTED_MODEL_PATH, "models/whisperwatch_boosted.joblib")
    for name in ("dev_predictions.csv", "dev_comparison.csv"):
        archive.write(BOOSTED_OUTPUT_DIR / name, f"outputs/boosted/{name}")
files.download(str(boosted_zip))